# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Display basic information
print(f"Dataset name: {getattr(metadata, 'name', None)}")
print(f"Description: {getattr(metadata, 'description', None)}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We will view all Record Sets, their `@id`, and all associated field `@id`s using `dataset.metadata`.

All IDs are referenced explicitly as required by the dataset schema.

In [ ]:
# Print all record sets and their field and column IDs
print("Record sets:")
for rs in getattr(metadata, 'record_sets', []):
    print(f"- Record Set: name={getattr(rs, 'name', None)}  @id={getattr(rs, '@id', None)}")
    fields = getattr(rs, 'fields', [])
    if fields:
        for field in fields:
            print(f"    Field: name={getattr(field, 'name', None)}  @id={getattr(field, '@id', None)} | Data type: {getattr(field, 'data_type', None)}")
            if hasattr(field, 'columns') and field.columns:
                for column in getattr(field, 'columns', []):
                    print(f"        Column: name={getattr(column, 'name', None)}  @id={getattr(column, '@id', None)}")
    print()
    

## 3. Data Extraction
Load data from all record sets into individual DataFrames for analysis.

We will reference each entity (record set, field, column) by its `@id`.

**Tip:** Use the printed overview above to select appropriate `@id` values when manipulating or visualizing data.

In [ ]:
# List all available RecordSet @id values
record_sets = [getattr(rs, '@id', None) for rs in getattr(metadata, 'record_sets', []) if getattr(rs, '@id', None)]
print(f"Available RecordSet @id's: {record_sets}")

# Create a dictionary to hold DataFrames for each RecordSet
dataframes = {}
for record_set_id in record_sets:
    # The public API: pass the correct @id to records
    records_gen = dataset.records(record_set=record_set_id)
    records = list(records_gen)
    dataframes[record_set_id] = pd.DataFrame(records)

# Preview the columns of each DataFrame
for record_set_id, df in dataframes.items():
    print(f"\nRecordSet @id: {record_set_id}")
    print("Columns:", df.columns.tolist())
    print(df.head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

> **All fields should be referenced by their exact `@id` as printed above.**

Below, we select a record set and a numeric field by their `@id`, filter records, normalize, and group by a categorical field (if it exists).

In [ ]:
# Example: Choose RecordSet and fields by their @id
# Please set these values according to your data overview above.
# e.g. record_set_id = 'cr:RecordSet_1', numeric_field_id = 'cr:coverage', group_field_id = 'cr:sample_condition'

# Example placeholders; replace with your actual @id values:
record_set_id = record_sets[0] if record_sets else None
# You may need to look at your DataFrame columns if they are verbose @id strings
df = dataframes[record_set_id]

# Preview all columns/@id values
print("Available columns for EDA (reference by @id):")
print(df.columns)

# Select sample numeric field and group field from the columns
numeric_field_id = None
group_field_id = None
for col in df.columns:
    # Simple heuristic: find numeric-looking fields (could be improved or made explicit)
    if numeric_field_id is None and pd.api.types.is_numeric_dtype(df[col]):
        numeric_field_id = col
    if group_field_id is None and not pd.api.types.is_numeric_dtype(df[col]):
        group_field_id = col
    if numeric_field_id and group_field_id:
        break

print(f"Using numeric field: {numeric_field_id}")
print(f"Using group field: {group_field_id}")

# Ensure there are no missing/nan before filtering
if numeric_field_id:
    # Filtering by a threshold (arbitrary, e.g. top 10% quantile if threshold is not known)
    threshold = df[numeric_field_id].quantile(0.90) if not df[numeric_field_id].empty else 0
    filtered_df = df[df[numeric_field_id] > threshold]

    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by group_field_id (if it's meaningful)
    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"Grouped data by {group_field_id}:")
        print(grouped_df.head())
else:
    print("No numeric field found for analysis.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

This example shows a histogram and a boxplot for the selected numeric field (referenced by its `@id`).

*Feel free to adapt or expand visualizations based on the data overview above.*

In [ ]:
import matplotlib.pyplot as plt

if numeric_field_id:
    # Histogram
    plt.figure(figsize=(8, 4))
    df[numeric_field_id].hist(bins=30)
    plt.title(f'Histogram of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()

    # Boxplot, optionally grouped
    plt.figure(figsize=(8, 4))
    if group_field_id and group_field_id in df.columns:
        df.boxplot(column=numeric_field_id, by=group_field_id, grid=False, rot=90)
        plt.title(f'Boxplot of {numeric_field_id} by {group_field_id}')
        plt.suptitle("")
    else:
        df.boxplot(column=numeric_field_id, grid=False)
        plt.title(f'Boxplot of {numeric_field_id}')
    plt.ylabel(numeric_field_id)
    plt.show()
else:
    print("No numeric field found for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- We successfully loaded and explored the FAIR^2 mass spectrometry dataset using the `mlcroissant` library.
- Record sets, fields, and columns were referenced using their `@id` fields for all extraction and processing steps, ensuring precise mapping to the dataset schema.
- Basic exploratory analysis was applied to a numeric column, with records filtered, normalized, and visualized.

Further analysis may include deeper feature engineering, modeling, or cross-referencing multiple record sets and their relationships.